## Imports

In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import itertools

In [2]:
# 1. 메타데이터 로드
with open("metadata.json", "r") as f:
    meta = json.load(f)

vocab_size = meta["vocab_size"]
max_len = meta["max_len"]
embed_dim = 64

# 2. 데이터 로드
df = pd.read_csv("llm_embedded_token_vector.csv")

## For DNN

In [10]:
# --- DNN 모델 정의 ---
class DNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_len, num_classes):
        super(DNN, self).__init__()

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=0)
        self.flatten_dim = max_len * embed_dim

        self.layer_stack = nn.Sequential(
            nn.Linear(self.flatten_dim, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.4),
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, num_classes)  ## 분류 문제에 따라 다른 수: 5/1
        ).to("cuda:0")
    def forward(self, x):
        x = self.embedding(x)
        x = torch.flatten(x, start_dim = 1, end_dim = -1)
        return self.layer_stack(x)  ## network

# --- 모델 훈련 및 평가 함수 ---
def train_and_evaluate(X, y, num_classes):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()   ## 피쳐 정규화
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.long)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.long).to("cuda:0")
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size = 64, shuffle=True)
    model = DNN(vocab_size = vocab_size, embed_dim = embed_dim, max_len = max_len, num_classes = num_classes).to("cuda:0")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

    ## 30 에폭 + GPU 배치 훈련
    num_epochs = 30 
    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in train_loader:
            inputs = inputs.to("cuda:0")
            labels = labels.to("cuda:0")
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        _, predicted = torch.max(outputs.data, 1)
        accuracy = accuracy_score(y_test, predicted.cpu().numpy())

    print(f"current model accurary: {accuracy:.4f}")
    return (predicted.cpu().numpy(), y_test) ## 정확도 반환... 라벨도 반환 해줘잉

In [11]:
torch.manual_seed(42)

# --- 다중 분류 실험 ---
le_multi = LabelEncoder()
y_multi_class = le_multi.fit_transform(df["model"])
num_classes_multi = len(np.unique(y_multi_class))

X = df.drop(["model"], axis = 1).values
multi_class_results = train_and_evaluate(X, y_multi_class, num_classes_multi)

current model accurary: 0.5558


In [12]:
# --- 이진 분류 실험 ---
df_binary = df.copy()
df_binary['model'] = df_binary['model'].apply(lambda x: 'human' if x == 'human' else 'llm')
le_binary = LabelEncoder()
y_binary_class = le_binary.fit_transform(df_binary['model'])
num_classes_binary = len(np.unique(y_binary_class))

X = df.drop(["model"], axis = 1).values
binary_class_results = train_and_evaluate(X, y_binary_class, num_classes_binary)

current model accurary: 0.9360


## 결과 종합

In [13]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
labels = ['deepseek', 'claude', 'gemini', 'gpt', 'human']
binary_labels = ["human", "llm"]

In [16]:
def heatmap_vis(y_true, y_pred, labels, type: str):
    plt.figure(figsize = (10, 8))
    ax = sns.heatmap(
        confusion_matrix(y_true, y_pred, labels = labels),
        annot = True,
        annot_kws = {"size": 20},
        fmt = "d",
        cmap = "Blues",
        xticklabels = labels,
        yticklabels = labels
    )

    ax.tick_params(axis = "x", top = True, bottom = False, labeltop = True, labelbottom = False)
    ax.set_title("Predicted Label", fontsize = 24, pad = 10)
    ax.set_ylabel("Actual Label", fontsize = 24)

    cb = ax.collections[0].colorbar
    cb.ax.tick_params(labelsize = 14)

    plt.xticks(fontsize = 20)
    plt.yticks(fontsize = 20, rotation = 0)
    plt.savefig(f"{type}_confusion_matrix.png", dpi = 300, bbox_inches = "tight")

def multi_get_report(y_true, y_pred):
    report_dict = classification_report(y_true, y_pred, labels = labels, output_dict = True)

    # pandas DataFrame으로 변환하여 출력 형식 지정
    df_report = pd.DataFrame(report_dict).transpose()
    print(df_report.to_string(float_format=lambda x: f"{x:.4f}"))

    return df_report

def binary_get_report(y_true, y_pred):
    # 이진 분류 리포트를 딕셔너리로 받기
    report_binary_dict = classification_report(y_true, y_pred, target_names=binary_labels, output_dict=True)

    # DataFrame으로 변환하여 출력 형식 지정
    df_binary_report = pd.DataFrame(report_binary_dict).transpose()
    print(df_binary_report.to_string(float_format=lambda x: f"{x:.4f}"))

    return df_binary_report

## 필요한 지표만 추출
def extract_use_metric(report, multi):
    if multi:
        acc = report.iloc[5, 0]
        bal_acc = recall = report.iloc[6, 1]
        precision = report.iloc[6, 0]
        f1 = report.iloc[6, 2]
    else:
        acc = report.iloc[2, 0]
        bal_acc = report.iloc[3, 0]
        precision = report.iloc[1, 0]
        recall = report.iloc[1, 1]
        f1 = report.iloc[1, 2]

    return np.array([acc, bal_acc, precision, recall, f1]).round(4)

`-` 주제별 결과 산출

In [23]:
print("="*15, "Multi-class Metrics", "="*15)
yyhat, yy = multi_class_results
pred, actual = le_multi.inverse_transform(yyhat), le_multi.inverse_transform(yy)

all_features_report_multi = multi_get_report(actual, pred)
extract_use_metric(all_features_report_multi, multi = True)

=============== Multi-class Metrics ===============
              precision  recall  f1-score   support
deepseek         0.4698  0.4667    0.4682  600.0000
claude           0.4720  0.4500    0.4608  600.0000
gemini           0.4042  0.4217    0.4127  600.0000
gpt              0.6250  0.7333    0.6748  600.0000
human            0.8657  0.7160    0.7838  567.0000
accuracy         0.5558  0.5558    0.5558    0.5558
macro avg        0.5673  0.5575    0.5601 2967.0000
weighted avg     0.5640  0.5558    0.5576 2967.0000


array([0.5558, 0.5575, 0.5673, 0.5575, 0.5601])

In [24]:
print("="*15, "Binary Metrics", "="*15)
yyhat, yy = binary_class_results
pred, actual = le_binary.inverse_transform(yyhat), le_binary.inverse_transform(yy)

all_features_report_binary = binary_get_report(actual, pred)
extract_use_metric(all_features_report_binary, multi = False)

=============== Binary Metrics ===============
              precision  recall  f1-score   support
human            0.9435  0.7072    0.8085  567.0000
llm              0.9347  0.9900    0.9616 2400.0000
accuracy         0.9360  0.9360    0.9360    0.9360
macro avg        0.9391  0.8486    0.8850 2967.0000
weighted avg     0.9364  0.9360    0.9323 2967.0000


array([0.936 , 0.9391, 0.9347, 0.99  , 0.9616])